# Data Description
- Author: JenniferjjLee
- Date first created: 2026-10-20
- Data Path: https://drive.google.com/drive/folders/1XwdYFkXrm0VOtCaPu-hkwpWAdY7RM-mW?usp=share_link
- Background: 文件描述 2024、2025 YouBike 票證刷卡交易資料，借車站或還車站任一端與研究清單精確匹配即保留（OR）。不進行去重、站名補字、車種推測或日期修正。兩年度分別驗證，不當成一份未區分年份的樣本。
- Unit of observation: 一筆來源交易紀錄；沒有交易 ID，不能宣稱七欄組合唯一。來源重複交易依原規則保留。
- Sample period: 依來源檔月份分為 202401–202412、202501–202512；不依還車日期重新分年。各時間欄實際最小/最大值與跨年情況見輸出。
- Known issues:
  - 參考清單 `stations_in_field_trver.csv` 有 81 列、78 個不同站名；三個重複名稱只作集合 membership，不倍增交易。這不是目前另有 89 列的 `data/processed/stations_in_field.csv`，不可混用版本。
  - 202401–202410 原始六欄缺少 bike_type，輸出空白，不代表一般車、0 或 Unknown。
  - 14 種異常站名依使用者確認為非目標；若另一端匹配，仍保留交易。來源已有的 `�` 保持原字串；無效 bytes 使用 `<UNRESOLVED_UTF8_HEX:...>` 可逆表示，並非正式站名或已修復文字。
  - 第二階段報告列 2024/2025 年另一端亂碼為 801/53 筆，但本次直接重讀年度 CSV 為 843/58 筆，差異 42/5 筆；不能宣稱既有旁表涵蓋所有異常交易。另一端空白為 1/3 筆。這些計數不是不同車站數。
  - 來源三筆 rent 格式異常均因兩端不匹配而未進年度檔，並非依時長刪除。格式合法也不表示日期或時長合理；本文件另報告還早於借、跨年還車、時長與整點時間差。
  - `rent_time` / `return_time` 在本資料呈整點格式，不能推定有分鐘或秒級事件精度；`rent` 為來源時長字串，衍生秒數只供描述，不覆寫原欄位。
- Definition for each variable:
  - rent_time：來源借車時間字串；衍生解析僅用於範圍與品質檢查。
  - rent_station：來源借車站名，逐字保留；可逆 token 須搭配旁表解讀。
  - return_time：來源還車時間字串，不因超出來源年度而刪除或重新歸年。
  - return_station：來源還車站名，處理同 rent_station。
  - rent：來源借用時長字串，常見格式為多位小時:兩位分鐘:兩位秒數；不等同直接可用的十進位小時。
  - bike_type：來源車種文字；早期未提供者為空字串。本文件把空字串計為缺失，不把 NA/NULL 等字面字串自動轉缺失。
  - infodate：來源交易日期；作月份核對，不視為檔案建立日期。
- Data path / Downstream analysis input（目前實際位置，屬衍生資料，雖存放於 raw 子資料夾）:
  - [YouBike_traffic_2024.csv](../raw/2024-25YouBike2.0票證刷卡資料/stage2_res/YouBike_traffic_2024.csv)：6,291,342 列 × 7 欄；SHA-256 `c5dc5f5cf3d37966f09a660b9454b09cfe251a54806cf38b491390c9f09f1859`。
  - [YouBike_traffic_2025.csv](../raw/2024-25YouBike2.0票證刷卡資料/stage2_res/YouBike_traffic_2025.csv)：8,252,926 列 × 7 欄；SHA-256 `35ddea00d0dfe9143d44b66724bdf5afd9ad58f73e75ee939152361c95e5ef74`。
- Source data: `data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/` 下 202401–202512 每月 `YYYYMM_YouBike2.0票證刷卡資料.csv`，以及同 YouBike 資料夾下的 `stations_in_field_trver.csv`；下方設定列出全部 24 個精確來源路徑。
- Processing code: [YouBike_traffic_raw.py](../raw/2024-25YouBike2.0票證刷卡資料/YouBike_traffic_raw.py)；可逆讀取機制來自同資料夾 [station_dataprocessing.py](../raw/2024-25YouBike2.0票證刷卡資料/station_dataprocessing.py)。本 Notebook 不執行處理程式、不改動 CSV。
- Role: 已按目標站名篩選的交易資料，供後續需求/流量分析；尚非全面修正日期、時長及另一端站名後的乾淨樣本。
- Verification status: 2026-10-20 以新建的 in-process IPython kernel 完整執行並保留輸出；兩年度 SHA、七欄、OR、逐月筆數與車種缺失規則通過。異常計數/旁表對帳未通過，品質狀態 needs_review；不能標記為全面清理完成。
- SHA-256: 各檔分別固定於以上說明及下方設定；檔案異動會停止讀取，不在執行時自動接受新版 hash。

In [1]:
from pathlib import Path
from collections import Counter
import hashlib, json, math
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows",100)
pd.set_option("display.max_colwidth",120)

def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('請從此學生專案內開啟 Notebook。')
PROJECT_ROOT = find_project_root()

def project_file(relative_path):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError('必須使用明確的專案相對路徑。')
    path = (PROJECT_ROOT / rel).resolve()
    if not path.is_relative_to((PROJECT_ROOT / 'data').resolve()) or not path.is_file():
        raise FileNotFoundError(path)
    return path

def sha256_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def verify_input_sha256(path, expected):
    actual = sha256_file(path)
    if actual != expected:
        raise ValueError(f'版本不同，請先核對，不自動接受新檔：{path}')
    return actual

## Change for Your Own Data

這裡已配置本次兩份實際資料，沒有範例資料或占位讀取器。固定七欄、UTF-8-SIG、全字串、空字串保留；按 100,000 列分批累積，不一次載入 1,454 萬筆交易。
原始月檔只列出處理血緣，不重跑第一、二階段。參考清單版本也先驗證 SHA，避免把新範圍混入舊交易。

In [2]:
BASE_REL = 'data/raw/2024-25YouBike2.0票證刷卡資料'
EXPECTED_COLUMNS = ['rent_time','rent_station','return_time','return_station','rent','bike_type','infodate']
INPUTS = {
    2024: {'path': f'{BASE_REL}/stage2_res/YouBike_traffic_2024.csv', 'rows':6291342,
           'sha256':'c5dc5f5cf3d37966f09a660b9454b09cfe251a54806cf38b491390c9f09f1859',
           'corrupted_other_rows':801, 'missing_other_rows':1},
    2025: {'path': f'{BASE_REL}/stage2_res/YouBike_traffic_2025.csv', 'rows':8252926,
           'sha256':'35ddea00d0dfe9143d44b66724bdf5afd9ad58f73e75ee939152361c95e5ef74',
           'corrupted_other_rows':53, 'missing_other_rows':3},
}
REFERENCE_REL = f'{BASE_REL}/stations_in_field_trver.csv'
REFERENCE_SHA256 = 'ea4d43a82df67636040a7c2792be97d8ad4aad773efd30d4f374ff1746476352'
reference_file = project_file(REFERENCE_REL)
verify_input_sha256(reference_file, REFERENCE_SHA256)
reference = pd.read_csv(reference_file, encoding='utf-8-sig', dtype='string', keep_default_na=False)
targets = set(reference.StationNameZh_tw)
assert len(reference)==81 and len(targets)==78
monthly_report = pd.read_csv(project_file(f'{BASE_REL}/stage2_res/stage2_data_quality_report.csv'),
                             dtype={'file_month':str})
assert len(monthly_report)==24 and monthly_report.processing_status.eq('validated').all()
SOURCE_INPUTS = [f'{BASE_REL}/raw_data/{y}{m:02}_YouBike2.0票證刷卡資料.csv'
                 for y in INPUTS for m in range(1,13)]
display(pd.DataFrame({'source_monthly_file': SOURCE_INPUTS}))
CHUNKSIZE = 100_000
TRACE_SHA256 = '4c7dc3b9b973e16a596e5a5598a4fb96c1acfc593748da9b4fc9519734c15651'
trace_file = project_file(f'{BASE_REL}/stage2_res/stage2_corrupted_station_records.csv')
verify_input_sha256(trace_file, TRACE_SHA256)
trace_report = pd.read_csv(trace_file,dtype={'file_month':str},keep_default_na=False)


,source_monthly_file
0,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202401_YouBike2.0票證刷卡資料.csv
1,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202402_YouBike2.0票證刷卡資料.csv
2,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202403_YouBike2.0票證刷卡資料.csv
3,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202404_YouBike2.0票證刷卡資料.csv
4,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202405_YouBike2.0票證刷卡資料.csv
5,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202406_YouBike2.0票證刷卡資料.csv
6,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202407_YouBike2.0票證刷卡資料.csv
7,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202408_YouBike2.0票證刷卡資料.csv
8,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202409_YouBike2.0票證刷卡資料.csv
9,data/raw/2024-25YouBike2.0票證刷卡資料/raw_data/202410_YouBike2.0票證刷卡資料.csv


## Summary

以下各年獨立重讀實際 CSV，核對 SHA-256、七欄及 OR 篩選條件。缺失定義為空字串或純空白；其他字面內容不當成缺失。
計數與日期解析只用於文件驗證；沒有篩除、去重、修名或回寫任何資料。

In [3]:
def suspicious_token(value):
    return value.startswith('<UNRESOLVED_UTF8_HEX:') or '\ufffd' in value

def inspect_year(year, config):
    path = project_file(config['path'])
    fingerprint = verify_input_sha256(path, config['sha256'])
    state = dict(rows=0, bad_or_rows=0, corrupted_other_rows=0, missing_other_rows=0,
                 rent_format_anomaly_rows=0, returned_before_rented_rows=0,
                 return_outside_source_year_rows=0, non_hour_aligned_time_rows=0,
                 rent_elapsed_difference_over_1h_rows=0)
    counts = {column:Counter() for column in EXPECTED_COLUMNS}
    missing = Counter(); months = Counter(); duration_seconds=Counter()
    invalid_dates=Counter(); ranges={}; sample=None; matched=set();corrupted_examples={}
    for chunk in pd.read_csv(path,encoding='utf-8-sig',dtype='string',keep_default_na=False,
                             chunksize=CHUNKSIZE,on_bad_lines='error'):
        assert list(chunk.columns)==EXPECTED_COLUMNS
        n=len(chunk);state['rows']+=n
        if sample is None:sample=chunk.head(10).copy()
        for col in EXPECTED_COLUMNS:
            frequencies=chunk[col].value_counts(dropna=False).to_dict()
            counts[col].update(frequencies)
            missing[col]+=sum(amount for value,amount in frequencies.items() if not value.strip())
        rent_match=chunk.rent_station.isin(targets);return_match=chunk.return_station.isin(targets)
        state['bad_or_rows']+=int((~(rent_match|return_match)).sum())
        matched.update(chunk.loc[rent_match,'rent_station']);matched.update(chunk.loc[return_match,'return_station'])
        bad_names={name for name in set(chunk.rent_station)|set(chunk.return_station) if suspicious_token(name)}
        corrupted_mask=chunk.rent_station.isin(bad_names)|chunk.return_station.isin(bad_names)
        state['corrupted_other_rows']+=int(corrupted_mask.sum())
        for index,row in chunk.loc[corrupted_mask].iterrows():
            corrupted_examples[int(index)+1]=row.to_dict()
        state['missing_other_rows']+=int((chunk.rent_station.str.strip().eq('')|chunk.return_station.str.strip().eq('')).sum())
        parsed={}
        for col in ['rent_time','return_time','infodate']:
            fmt='%Y-%m-%d' if col=='infodate' else '%Y-%m-%d %H:%M:%S'
            value=pd.to_datetime(chunk[col],format=fmt,errors='coerce')
            parsed[col]=value;invalid_dates[col]+=int(value.isna().sum())
            if value.notna().any():
                low,high=value.min(),value.max()
                old=ranges.get(col,(low,high));ranges[col]=(min(old[0],low),max(old[1],high))
        months.update(chunk.infodate.str[:7].value_counts().to_dict())
        state['returned_before_rented_rows']+=int((parsed['return_time']<parsed['rent_time']).sum())
        state['return_outside_source_year_rows']+=int((parsed['return_time'].notna() & parsed['return_time'].dt.year.ne(year)).sum())
        aligned = (parsed['rent_time'].dt.minute.eq(0)&parsed['rent_time'].dt.second.eq(0)
                   &parsed['return_time'].dt.minute.eq(0)&parsed['return_time'].dt.second.eq(0))
        state['non_hour_aligned_time_rows']+=int((~aligned).sum())
        good=chunk.rent.str.fullmatch(r'\d+:[0-5]\d:[0-5]\d',na=False)
        state['rent_format_anomaly_rows']+=int((~good).sum())
        if good.any():
            parts=chunk.loc[good,'rent'].str.split(':',expand=True).astype('int64')
            seconds=parts[0]*3600+parts[1]*60+parts[2]
            duration_seconds.update(seconds.value_counts().to_dict())
            elapsed=(parsed['return_time']-parsed['rent_time']).dt.total_seconds()
            state['rent_elapsed_difference_over_1h_rows']+=int((seconds.sub(elapsed.loc[good]).abs()>3600).sum())
        if state['rows']%1_000_000==0:print(f'{year}: 已檢查 {state["rows"]:,} 列',flush=True)
    state.update(year=year,sha256=fingerprint,schema={c:'string（空字串保留）' for c in EXPECTED_COLUMNS},
                 data_path=config['path'],matched_target_stations=len(matched))
    return dict(summary=state,counts=counts,missing=missing,months=months,sample=sample,
                invalid_dates=invalid_dates,ranges=ranges,duration_seconds=duration_seconds,
                unmatched_targets=sorted(targets-matched),corrupted_examples=corrupted_examples)

results={year:inspect_year(year,config) for year,config in INPUTS.items()}
overview=pd.DataFrame([result['summary'] for result in results.values()]).set_index('year')
display(overview.drop(columns=['schema','sha256','data_path']).T)
for year,result in results.items():
    print(year, result['summary']['data_path'], result['summary']['sha256'])
    display(pd.DataFrame([{'column':col,'dtype':'string','min':str(pair[0]),'max':str(pair[1]),
                           'invalid_datetime_rows':result['invalid_dates'][col]}
                          for col,pair in result['ranges'].items()]))

year,2024,2025
rows,6291342,8252926
bad_or_rows,0,0
corrupted_other_rows,843,58
missing_other_rows,1,3
rent_format_anomaly_rows,0,0
returned_before_rented_rows,0,0
return_outside_source_year_rows,187,50
non_hour_aligned_time_rows,0,0
rent_elapsed_difference_over_1h_rows,4,6
matched_target_stations,78,78


,column,dtype,min,max,invalid_datetime_rows
0,rent_time,string,2024-01-01 00:00:00,2024-12-31 23:00:00,0
1,return_time,string,2024-01-01 00:00:00,2034-08-26 10:00:00,0
2,infodate,string,2024-01-01 00:00:00,2024-12-31 00:00:00,0


,column,dtype,min,max,invalid_datetime_rows
0,rent_time,string,2025-01-01 00:00:00,2025-12-31 23:00:00,0
1,return_time,string,2025-01-01 00:00:00,2026-01-01 06:00:00,0
2,infodate,string,2025-01-01 00:00:00,2025-12-31 00:00:00,0


### Sample

每年各列出實際前十筆。表內空白 bike_type 仍是原檔空字串，並未變成 0 或一般車。

In [4]:
for year,result in results.items():
    print(f'{year} 年：前 10 筆（原始字串）')
    display(result['sample'])

,rent_time,rent_station,return_time,return_station,rent,bike_type,infodate
0,2024-01-22 08:00:00,新生和平路口東北側,2024-01-22 09:00:00,新生南路三段66號前,00:07:02,,2024-01-22
1,2024-01-22 12:00:00,臺大小福樓東側,2024-01-22 12:00:00,捷運臺電大樓站(5號出口),00:12:21,,2024-01-22
2,2024-01-22 21:00:00,捷運公館站(2號出口),2024-01-22 22:00:00,臺灣科技大學側門,00:19:06,,2024-01-22
3,2024-01-22 23:00:00,捷運公館站(2號出口),2024-01-23 00:00:00,臺灣科技大學側門,00:26:58,,2024-01-22
4,2024-01-22 18:00:00,臺大工綜館南側,2024-01-22 18:00:00,臺大椰林小舖,00:04:00,,2024-01-22
5,2024-01-22 15:00:00,臺大新生教室南側,2024-01-22 15:00:00,臺大新生教室南側,00:23:07,,2024-01-22
6,2024-01-22 17:00:00,臺大卓越研究大樓,2024-01-22 17:00:00,臺大黑森林西側,00:11:16,,2024-01-22
7,2024-01-18 11:00:00,林森仁愛路口,2024-01-18 11:00:00,臺大第一活動中心西南側,00:22:55,,2024-01-18
8,2024-01-18 16:00:00,林森仁愛路口,2024-01-18 16:00:00,新生南路三段82號前,00:22:05,,2024-01-18
9,2024-01-18 16:00:00,林森仁愛路口,2024-01-18 16:00:00,臺大第一活動中心西南側,00:23:21,,2024-01-18


,rent_time,rent_station,return_time,return_station,rent,bike_type,infodate
0,2025-01-13 15:00:00,捷運古亭站(8號出口),2025-01-13 21:00:00,新生南路三段52號前,05:34:25,電輔車,2025-01-13
1,2025-01-13 01:00:00,辛亥路一段30號前,2025-01-13 03:00:00,捷運公館站(4號出口),01:50:15,一般車,2025-01-13
2,2025-01-13 12:00:00,臺大男一舍前,2025-01-13 12:00:00,捷運公館站(2號出口),00:06:22,一般車,2025-01-13
3,2025-01-13 15:00:00,臺大卓越研究大樓,2025-01-13 16:00:00,臺大立體機車停車場,00:35:58,一般車,2025-01-13
4,2025-01-13 15:00:00,捷運公館站(2號出口),2025-01-13 16:00:00,臺大鹿鳴堂東側,00:32:42,一般車,2025-01-13
5,2025-01-13 15:00:00,臺大小福樓東側,2025-01-13 16:00:00,臺大小福樓東側,00:38:56,一般車,2025-01-13
6,2025-01-13 13:00:00,臺大工綜館南側,2025-01-13 13:00:00,臺大二號館,00:03:08,一般車,2025-01-13
7,2025-01-13 18:00:00,大安運動中心,2025-01-13 18:00:00,捷運科技大樓站(台北教育大學,00:03:48,一般車,2025-01-13
8,2025-01-13 22:00:00,捷運公館站(3號出口),2025-01-13 22:00:00,臺大社科院西側,00:04:38,一般車,2025-01-13
9,2025-01-13 17:00:00,臺大新生教室南側,2025-01-13 18:00:00,捷運萬隆站(4號出口),00:42:20,一般車,2025-01-13


### Summary Stats

七個原欄位都保留為字串：分別列出有效/缺失 N、百分比、Unique 值與前十類次數。缺失比例與前十類比例的分母都是該年度全部交易筆數，前十類不包含空值。
另外依 rent 原字串計算「來源時長秒數」的完整加權摘要，分位數採排序觀測值的線性插值，標準差採樣本標準差（ddof=1）；不是只抽樣估計，也不是用時間戳差替換 rent。

In [5]:
def weighted_duration_summary(counter):
    if not counter:return {'N':0}
    values=np.array(sorted(counter),dtype=np.float64)
    weights=np.array([counter[int(value)] for value in values],dtype=np.int64)
    cumulative=weights.cumsum();n=int(weights.sum())
    mean=math.fsum(float(v)*int(w) for v,w in zip(values,weights))/n
    def quantile(q):
        position=(n-1)*q;left=math.floor(position);right=math.ceil(position)
        a=values[np.searchsorted(cumulative,left,side='right')]
        b=values[np.searchsorted(cumulative,right,side='right')]
        return float(a+(b-a)*(position-left))
    variance=math.fsum(int(w)*(float(v)-mean)**2 for v,w in zip(values,weights))/(n-1) if n>1 else np.nan
    return dict(N=n,mean=mean,min=float(values[0]),q25=quantile(.25),q50=quantile(.5),
                q75=quantile(.75),max=float(values[-1]),sd=math.sqrt(variance))

for year,result in results.items():
    n=result['summary']['rows']
    print(f'{year} 年；比例分母 N={n:,}')
    display(pd.DataFrame([dict(variable=col,N=n,nonmissing=n-result['missing'][col],
        missing=result['missing'][col],missing_pct=result['missing'][col]/n*100,
        unique_nonmissing=sum(bool(value.strip()) for value in result['counts'][col]))
        for col in EXPECTED_COLUMNS]))
    top=[]
    for col in EXPECTED_COLUMNS:
        valid=[(value,count) for value,count in result['counts'][col].most_common() if value.strip()][:10]
        top.extend(dict(variable=col,rank=i,value=value,count=count,pct_of_all_rows=count/n*100)
                   for i,(value,count) in enumerate(valid,1))
    display(pd.DataFrame(top))
    print('來源 rent 解析秒數摘要（非交易時間戳差）')
    display(pd.DataFrame([weighted_duration_summary(result['duration_seconds'])]))
monthly_summary=pd.DataFrame([dict(year=year,source_month=month,rows=amount)
    for year,result in results.items() for month,amount in sorted(result['months'].items())])
display(monthly_summary)

,variable,N,nonmissing,missing,missing_pct,unique_nonmissing
0,rent_time,6291342,6291342,0,0.000000,8656
1,rent_station,6291342,6291342,0,0.000000,1277
2,return_time,6291342,6291342,0,0.000000,8684
3,return_station,6291342,6291341,1,0.000016,2016
4,rent,6291342,6291342,0,0.000000,17965
5,bike_type,6291342,1455401,4835941,76.866605,2
6,infodate,6291342,6291342,0,0.000000,364


,variable,rank,value,count,pct_of_all_rows
0,rent_time,1,2024-09-11 12:00:00,3918,0.062276
1,rent_time,2,2024-09-09 17:00:00,3894,0.061895
2,rent_time,3,2024-09-25 12:00:00,3858,0.061322
3,rent_time,4,2024-09-20 12:00:00,3830,0.060877
4,rent_time,5,2024-09-18 12:00:00,3776,0.060019
5,rent_time,6,2024-09-24 12:00:00,3753,0.059653
6,rent_time,7,2024-09-19 12:00:00,3715,0.059049
7,rent_time,8,2024-09-03 12:00:00,3699,0.058795
8,rent_time,9,2024-09-12 12:00:00,3674,0.058398
9,rent_time,10,2024-09-04 12:00:00,3628,0.057667


,N,mean,min,q25,q50,q75,max,sd
0,6291342,765.463201,0.0,276.0,423.0,749.0,315533109.0,125860.043197


,variable,N,nonmissing,missing,missing_pct,unique_nonmissing
0,rent_time,8252926,8252926,0,0.000000,8760
1,rent_station,8252926,8252926,0,0.000000,1445
2,return_time,8252926,8252926,0,0.000000,8764
3,return_station,8252926,8252923,3,0.000036,2335
4,rent,8252926,8252926,0,0.000000,17976
5,bike_type,8252926,8252926,0,0.000000,2
6,infodate,8252926,8252926,0,0.000000,365


,variable,rank,value,count,pct_of_all_rows
0,rent_time,1,2025-09-10 12:00:00,4424,0.053605
1,rent_time,2,2025-09-03 12:00:00,4340,0.052587
2,rent_time,3,2025-09-09 12:00:00,4275,0.051800
3,rent_time,4,2025-09-10 17:00:00,4264,0.051667
4,rent_time,5,2025-09-04 12:00:00,4208,0.050988
5,rent_time,6,2025-09-08 17:00:00,4201,0.050903
6,rent_time,7,2025-09-11 12:00:00,4196,0.050843
7,rent_time,8,2025-09-30 17:00:00,4171,0.050540
8,rent_time,9,2025-09-02 12:00:00,4138,0.050140
9,rent_time,10,2025-09-11 17:00:00,4134,0.050091


,N,mean,min,q25,q50,q75,max,sd
0,8252926,652.341357,0.0,263.0,405.0,705.0,1559155.0,2250.09384


,year,source_month,rows
0,2024,2024-01,238559
1,2024,2024-02,243087
2,2024,2024-03,435780
3,2024,2024-04,448208
4,2024,2024-05,526574
5,2024,2024-06,381407
6,2024,2024-07,519331
7,2024,2024-08,589940
8,2024,2024-09,785899
9,2024,2024-10,667156


### Assertions

只驗證已有依據的性質：檔案版本、筆數、七欄、OR 條件、每月來源對應、車種缺失規則、另一端異常計數的對帳結果（不一致會明確標 needs_review）及 78 個目標站涵蓋。
不要求沒有重複交易，不宣稱存在唯一鍵，也不把「還車日期超出來源年度」直接當成應刪除資料。

In [6]:
checks=[]
for year,result in results.items():
    info=INPUTS[year];state=result['summary'];n=state['rows']
    assert n==info['rows'] and state['bad_or_rows']==0
    corruption_count_agrees=(state['corrupted_other_rows']==info['corrupted_other_rows'])
    assert state['missing_other_rows']==info['missing_other_rows']
    assert state['matched_target_stations']==78 and not result['unmatched_targets']
    assert all(sum(result['counts'][col].values())==n for col in EXPECTED_COLUMNS)
    expected_months=monthly_report.loc[monthly_report.file_month.str.startswith(str(year))]
    assert len(expected_months)==12
    for row in expected_months.itertuples():
        month=f'{row.file_month[:4]}-{row.file_month[4:]}'
        assert result['months'].get(month,0)==row.filtered_rows
    assert sum(result['months'].values())==n
    expected_missing=int(expected_months.filtered_missing_bike_type_rows.sum())
    assert result['missing']['bike_type']==expected_missing
    assert state['rent_format_anomaly_rows']==int(expected_months.filtered_rent_format_anomaly_rows.sum())
    assert sum(result['invalid_dates'].values())==0
    actual=verify_input_sha256(project_file(info['path']),info['sha256'])
    recorded=set(trace_report.loc[trace_report.file_month.str.startswith(str(year)) & trace_report.issue_type.eq('corrupted_other_station'),'annual_output_row'])
    detected=set(result['corrupted_examples'])
    assert not (recorded-detected), '旁表存在不符合年度異常偵測的紀錄'
    missing_from_trace=sorted(detected-recorded)
    print(f'{year}：年度 CSV 異常 {len(detected)} 筆，既有旁表 {len(recorded)} 筆，漏列 {len(missing_from_trace)} 筆')
    display(pd.DataFrame([dict(year=year,annual_output_row=number,**result['corrupted_examples'][number]) for number in missing_from_trace[:3]]))
    checks.append(dict(year=year,rows=n,columns=7,OR_passed=True,month_counts_passed=True,
                       reported_corrupted_rows=info['corrupted_other_rows'],
                       observed_corrupted_rows=state['corrupted_other_rows'],
                       corruption_count_agrees=corruption_count_agrees,
                       missing_from_trace_rows=len(missing_from_trace),sha256=actual,
                       status='core checks passed' if corruption_count_agrees else 'needs_review: corruption report mismatch'))
assert verify_input_sha256(reference_file,REFERENCE_SHA256)==REFERENCE_SHA256
display(pd.DataFrame(checks))
print('筆數、七欄、OR、月份與 SHA 核對完成；另一端異常計數與第二階段報告不一致，品質對帳未通過，資料標記 needs_review。沒有修改或清理原始 CSV。')

,year,annual_output_row,rent_time,rent_station,return_time,return_station,rent,bike_type,infodate
0,2024,2589164,2024-07-28 10:00:00,基隆長興路口,2024-07-28 10:00:00,<UNRESOLVED_UTF8_HEX:e58391e5ae89e59cb0e4b88be5819ce8bb8ae5a0b42832e8999fe587bae58fa329e69db1e58d97e5>,00:14:52,,2024-07-28
1,2024,2824917,2024-08-02 18:00:00,臺大鹿鳴堂東側,2024-08-02 20:00:00,<UNRESOLVED_UTF8_HEX:e696b0e58c97e5b882e694bfe5ba9ce69dbfe6a98be58886e5b180e6b299e5b499e6b4bee587bae6>,01:28:16,,2024-08-02
2,2024,2828893,2024-08-28 06:00:00,<UNRESOLVED_UTF8_HEX:e58391e5ae89e59cb0e4b88be5819ce8bb8ae5a0b42832e8999fe587bae58fa329e69db1e58d97e5>,2024-08-28 06:00:00,大安運動中心,00:16:46,,2024-08-28


,year,annual_output_row,rent_time,rent_station,return_time,return_station,rent,bike_type,infodate
0,2025,82791,2025-01-11 15:00:00,辛亥新生路口東南側,2025-01-11 15:00:00,<UNRESOLVED_UTF8_HEX:e58391e5ae89e59cb0e4b88be5819ce8bb8ae5a0b42832e8999fe587bae58fa329e69db1e58d97e5>,00:45:45,一般車,2025-01-11
1,2025,90426,2025-01-22 08:00:00,<UNRESOLVED_UTF8_HEX:e58391e5ae89e59cb0e4b88be5819ce8bb8ae5a0b42832e8999fe587bae58fa329e69db1e58d97e5>,2025-01-22 08:00:00,新生南路三段94巷口,00:46:56,一般車,2025-01-22
2,2025,133379,2025-01-03 18:00:00,<UNRESOLVED_UTF8_HEX:e58391e5ae89e59cb0e4b88be5819ce8bb8ae5a0b42832e8999fe587bae58fa329e69db1e58d97e5>,2025-01-03 19:00:00,新生南路三段52號前,00:31:28,一般車,2025-01-03


,year,rows,columns,OR_passed,month_counts_passed,reported_corrupted_rows,observed_corrupted_rows,corruption_count_agrees,missing_from_trace_rows,sha256,status
0,2024,6291342,7,True,True,801,843,False,42,c5dc5f5cf3d37966f09a660b9454b09cfe251a54806cf38b491390c9f09f1859,needs_review: corruption report mismatch
1,2025,8252926,7,True,True,53,58,False,5,35ddea00d0dfe9143d44b66724bdf5afd9ad58f73e75ee939152361c95e5ef74,needs_review: corruption report mismatch


### 本次文件核對的限制

年度 CSV 與固定 SHA 相符，筆數、七欄、OR 與各月保留數可核對；但既有異常計數及旁表少了 47 筆（2024：42；2025：5）。文件顯示實際數字與漏列的年度行號範例，將資料品質標為 **needs_review**，沒有把來源報告改成表面通過。

本次僅更新文件及資料索引，不回寫年度 CSV、不修補第二階段程式或旁表。造成先前漏列的程式原因仍待另案排查；若後續分析依賴另一端站名品質或旁表完整性，須先處理此差異。2034 年還車日期及時長差異也保留為需核對事項。


| 指標 | 2024 | 2025 |
| --- | ---: | ---: |
| 交易列數 | 6,291,342 | 8,252,926 |
| 另一端異常（CSV直接重讀） | 843 | 58 |
| 另一端空白 | 1 | 3 |
| 還車年份超出來源年 | 187 | 50 |
| 來源時長與整點時間差逾1小時 | 4 | 6 |

2024 年 bike_type 空白 4,835,941 筆（76.87%）；2025 年無空白車種。

來源時長中位數為 2024 年 423 秒、2025 年 405 秒。2024 年最大來源時長 315,533,109 秒，且還車日期最晚到 2034-08-26；平均數與標準差包含這些原始極端值，不能解讀成已清理後的旅次時間分布。超出來源年度的還車包含正常跨年情況，不全部視為資料錯誤。
